# Training runs on Kaggle

My laptop GPU is small, so the training runs happen here on a free Kaggle T4
instead. Same scripts and configs as a normal run, everything logs to the
PlantDisease-CW project on ClearML.

Before running:

1. Upload the repo zip as a Kaggle dataset. On github.kcl.ac.uk, **switch the
   branch selector to `dev` first**, then Code -> Download ZIP (the button
   downloads whatever branch is on screen, and the default is master, which
   has the old data pipeline). Then Datasets -> New Dataset on Kaggle.
   In the dataset description, record the code version, e.g.
   "dev @ 1d04e41, downloaded 17 Nov 2026" plus the fingerprint cell 1 prints.
2. Session options: accelerator GPU (T4 x2 is fine, the script uses one GPU),
   internet ON.
3. Add-ons -> Secrets: CLEARML_API_ACCESS_KEY and CLEARML_API_SECRET_KEY,
   with the values from your clearml.conf. Do not paste them into a cell.

Cell 1 checks the zip is actually the dev pipeline and refuses to train
otherwise. Timings below are what the 15 Nov runs actually took on a T4 with
early stopping: comparison runs about 5 min each, full pool about 6 min (CNN)
and 15 min (ResNet18).


In [ ]:
# find the repo in the dataset input, copy it somewhere writable,
# then verify which pipeline this zip actually is
import os, shutil, zipfile, hashlib, yaml
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "leaf-image-plant-disease-classifier-CW"

if not (REPO / "scripts" / "train.py").exists():
    # already-extracted dataset folder?
    found = (list(INPUT.glob("**/leaf-image-plant-disease-classifier-CW/scripts/train.py"))
             or list(INPUT.glob("**/scripts/train.py")))
    if found:
        shutil.copytree(found[0].parents[1], REPO, dirs_exist_ok=True)
    else:
        # raw zip (in case Kaggle did not auto-extract)
        zips = list(INPUT.glob("**/*.zip"))
        assert zips, "Add the repo zip as a Kaggle Dataset (see checklist), then re-run."
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(WORK)
        found = list(WORK.glob("**/scripts/train.py"))
        assert found, "repo scripts not found after extraction"
        REPO = found[0].parents[1]

os.chdir(REPO)
print("repo:", REPO)
!ls

# Provenance guard: the 15 Nov runs used a master zip by mistake (no
# normalisation / augmentation pipeline). This refuses to train on that code.
_cfg = yaml.safe_load(open("configs/default.yaml"))
assert "transformations" in _cfg.get("data", {}), (
    "This zip has no data.transformations: it is the master pipeline, not dev. "
    "Re-download the ZIP from the dev branch and re-upload the dataset.")
_src = Path("scripts/train.py").read_text(encoding="utf-8")
assert "class_names" in _src, (
    "scripts/train.py does not upload class_names: it is not the dev pipeline. "
    "Re-download the ZIP from the dev branch and re-upload the dataset.")

print("provenance OK: dev pipeline")
print("code fingerprint (copy into the Kaggle dataset description):")
for _f in ("scripts/train.py", "src/plantdisease/train.py",
           "configs/default.yaml", "configs/transfer.yaml"):
    _h = hashlib.sha256(Path(_f).read_bytes()).hexdigest()[:16]
    print("  %s: %s" % (_f, _h))


In [ ]:
# deps (torch/torchvision already on Kaggle)
!pip install -q -r requirements.txt

In [ ]:
# ClearML creds from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["CLEARML_API_ACCESS_KEY"] = secrets.get_secret("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = secrets.get_secret("CLEARML_API_SECRET_KEY")
print("ClearML credentials loaded")

In [ ]:
# sanity check
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable GPU: Session options -> Accelerator -> GPU, then re-run"


## Comparison runs

Both configs at their defaults: 25% prototype pool, augmentation on, AdamW,
early stopping on validation accuracy (patience 3).

CNN: 30 epochs, batch 32, 128px. ResNet18: 15 epochs, batch 64, 224px.

The numbers from these runs are what the model choice is based on.


In [ ]:
# CNN from scratch, 25% pool (~5 min)
!python scripts/train.py --config configs/default.yaml --num-workers 4

In [ ]:
# ResNet18 transfer, 25% pool (~5 min)
!python scripts/train.py --config configs/transfer.yaml --num-workers 4

In [ ]:
# check the runs landed in ClearML
from clearml import Task

tasks = Task.get_tasks(project_name="PlantDisease-CW")
for t in sorted(tasks, key=lambda x: x.data.created):
    print("%s  %-10s  %s  %s" % (t.id, t.data.status, str(t.data.created)[:10], t.name))

## Full pool finals

Uncomment one cell at a time. With early stopping these took about 6 min (CNN)
and 15 min (ResNet18) on a T4. Tag the runs that go in the report
final-benchmark in ClearML and the throwaway ones scratch-ignore. The
comparison runs above count as scratch unless we decide otherwise.


In [ ]:
# full pool CNN (~6 min)
# !python scripts/train.py --config configs/default.yaml --full-pool --num-workers 4

In [ ]:
# full pool ResNet18 (~15 min)
# !python scripts/train.py --config configs/transfer.yaml --full-pool --num-workers 4

## If something breaks

- Provenance assert fails: the zip is the master pipeline. Re-download the ZIP
  from the dev branch (switch the branch selector before clicking Download
  ZIP), re-upload the dataset, restart the session.
- ClearML auth error: check the secret names match exactly.
- GPU assert fails: turn the accelerator on, that restarts the session, then
  run all again.
- Out of memory: add --batch-size 16 to the failing command.
- Session died mid run: rerun that cell. The dataset redownloads and ClearML
  gets a new task, tag the dead one scratch-ignore.
- Repo zip not found: the dataset is not attached, right panel -> Input ->
  Add Input.
